# evaluate — Direction 1: a policy in the organisers' simulator, against the reference runs

Runs a policy for 30 days (`hvac_controller/d1/evaluate.py`), one fresh process per scenario × seed, and compares
it per scenario with the reference runs in `buildings` (K10 and V1).

- `POLICY = "pkg"`: the attached `hvac-submission` package, run with its own `d1_policy.py`; its results are named
  `TAG`. `POLICY = "k10"`: the public baseline.
- `SCENARIOS`, comma-separated: `held_out` (19: building_train_2's 5 shipped strips, building_train_2 × 2 new
  stations, 6 held-out generated buildings × 2 strips), `shipped` (30), `generated` (all 86 of `manifest.json`;
  replica only, since the training strips' weather presets are not in the dataset), `stress` (25 of
  `manifest_stress.json`: held-out buildings at harsh strips of new stations, data.ipynb `stress`). `ONLY` keeps
  just the listed scenarios. Buildings above `MAX_ZONES` are left out.
- `WHERE`: `original` (the truth) and / or `replica` (the NumPy replica, from the caches).
- `SEEDS`: seed 0 sees its observations as they are; seeds ≥ 1 add 1e-6 noise to the state observations (the
  closed loop is chaotic, D1 doc §2.7).
- The jobs are sorted longest first (times from the reference runs) and dealt to `N_SHARDS` sessions. One 30-day
  run of an 85-zone building takes ≈ 35 min with 4 at once (building_train_2), so 2 sessions finish the 19 in ≈ 2 h.

The setup also compares the organisers' simulator on the competition mount (what the hidden run uses) with the
repo's copy, code without comments and docstrings, and prints the files that differ. Known: `building_utils.py` and
the two `constants.py` differ only in debug paths (2026-10-16 copy).

**Inputs:** the competition data, `buildings`, and `hvac-submission` for `POLICY = "pkg"`. CPU, Internet on.

**Output:** `/kaggle/working/score_<TAG>_<run>.zip`: one JSON per run; the comparison is printed.

In [ ]:
REPO_URL = "https://github.com/CryAndRRich/hvac-controller"
REPO_REF = "direction1"
POLICY, TAG = "pkg", "v2.2"  # "pkg" (hvac-submission, named TAG) or "k10"
SCENARIOS = "held_out"     # comma-separated: "held_out" (19), "shipped" (30), "generated" (86), "stress" (25)
ONLY = ""                  # comma-separated building/weather names to keep; "" keeps all
WHERE, SEEDS = "original", 1
RUN_ID, N_SHARDS = 0, 2
N_PROCS = 4                # simulator runs at once
MAX_ZONES = 110            # as train.ipynb: larger generated buildings are left out

In [ ]:
import ast, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

REPO, INPUT, WORK = Path("/tmp/hvac-controller"), Path("/kaggle/input"), Path("/tmp/work")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_REF], check=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
sys.path.insert(0, str(REPO))
ENV = dict(os.environ, PYTHONPATH=str(REPO))


def dataset(slug):
    """One of the four datasets (docs/KAGGLE.md), wherever Kaggle mounts it."""
    return next(p for p in (INPUT / "datasets" / "cryandrrich" / slug, INPUT / slug) if p.is_dir())


SHIPPED = next(p for p in sorted(INPUT.rglob("training_scenarios"))
               if (p / "building_train_1" / "atlanta_2" / "config.gin").exists())
ROOT = REPO / "smart_buildings" / "training_scenarios"
shutil.copytree(SHIPPED, ROOT, dirs_exist_ok=True)


def code(path):
    """A module's code without comments and docstrings."""
    tree = ast.parse(path.read_text())
    for node in ast.walk(tree):
        body = getattr(node, "body", None)
        if (isinstance(body, list) and body and isinstance(body[0], ast.Expr)
                and isinstance(body[0].value, ast.Constant) and isinstance(body[0].value.value, str)):
            node.body = body[1:] or [ast.Pass()]
    return ast.dump(tree)


MOUNTED = SHIPPED.parent  # the organisers' simulator package that the hidden run uses
pairs = [(p, REPO / "smart_buildings" / p.relative_to(MOUNTED)) for p in sorted(MOUNTED.rglob("*.py"))
         if "training_scenarios" not in p.parts]
print("simulator code that differs between the competition mount and the repo:",
      [str(p.relative_to(MOUNTED)) for p, q in pairs if q.exists() and code(p) != code(q)] or "none")
DATA = dataset("buildings")
if POLICY == "pkg":
    PKG = WORK / TAG  # the package, run with its own code; the folder names its results
    shutil.copytree(dataset("hvac-submission"), PKG, dirs_exist_ok=True)
    print((PKG / "README.txt").read_text())
    POLICIES = f"pkg:{PKG}"
else:
    POLICIES = TAG = "k10"

shipped = sorted(f"{p.parent.parent.name}/{p.parent.name}" for p in ROOT.glob("*/*/config.gin"))
manifest = json.loads((DATA / "manifest.json").read_text())
stress = json.loads((DATA / "manifest_stress.json").read_text()) if (DATA / "manifest_stress.json").exists() else []
held_out = {e["building"] for e in manifest if e["tag"] == "shipped_building_new_station"}  # building_train_2
name = lambda e: e["label"].replace("__", "/", 1)
SETS = {"shipped": shipped, "generated": [name(e) for e in manifest], "stress": [name(e) for e in stress],
        "held_out": [s for s in shipped if s.split("/")[0] in held_out]
                    + [name(e) for e in manifest if e["tag"] in ("gen_val", "shipped_building_new_station")]}
LIST = sorted({s for k in SCENARIOS.split(",") for s in SETS[k]})
if ONLY:
    LIST = [s for s in LIST if s in ONLY.split(",")]
assert LIST, "no scenario selected"
if "original" in WHERE:  # the generated ones are rebuilt from the dataset's YAMLs and weather presets
    assert not {name(e) for e in manifest if e["tag"] == "gen_train"} & set(LIST), "gen_train: replica only"
    for p in DATA.glob("building_[0-9]*.yaml"):
        shutil.copy(p, REPO / "smart_buildings" / "buildings" / p.name)
    (REPO / "smart_buildings" / "weather" / "csv").mkdir(parents=True, exist_ok=True)
    for pattern in ("[xs][0-9][0-9][0-9].yaml", "[xs][0-9][0-9][0-9].csv"):
        for p in DATA.glob(pattern):
            shutil.copy(p, REPO / "smart_buildings" / "weather" / ("csv" if p.suffix == ".csv" else "") / p.name)
    for path, tags in (("manifest.json", "gen_val,shipped_building_new_station"), ("manifest_stress.json", "stress")):
        if (DATA / path).exists():
            r = subprocess.run([sys.executable, "-m", "hvac_controller.d1.augment", "scenarios", str(DATA / path),
                                "--tags", tags], cwd=REPO, env=ENV, capture_output=True, text=True)
            print(f"{path} scenario folders: exit", r.returncode, (r.stdout + r.stderr)[-1500:])
            assert r.returncode == 0
ZONES = {p.stem.replace("__", "/", 1): json.loads(p.read_text())["n_zones"] for p in DATA.glob("building_*__*.json")}
LIST = [s for s in LIST if ZONES[s] <= MAX_ZONES]
REFS = [json.loads(p.read_text()) for p in sorted(DATA.glob("original__*.json")) + sorted(DATA.glob("replica__*.json"))]
seconds = {}  # of each scenario's 30 days in the original simulator, from the reference runs
for r in REFS:
    if r["where"] == "original":
        seconds[r["scenario"]] = max(seconds.get(r["scenario"], 0.0), r["seconds"])
for s in LIST:  # no reference run: ≈ 30 s per zone (building_train_2: 24, V2's 203-zone buildings: 34-41)
    seconds.setdefault(s, 30.0 * ZONES[s])
HERE = sorted(LIST, key=seconds.get, reverse=True)[RUN_ID::N_SHARDS]
print(f"{len(HERE)} of {len(LIST)} scenarios here, the longest {max(seconds[s] for s in HERE) / 3600:.1f} h:", HERE)

In [ ]:
# The runs, N_PROCS at a time; each result is printed as it ends.
EVAL = WORK / "eval"
cmd = [sys.executable, "-m", "hvac_controller.d1.evaluate", str(EVAL), "--scenarios", ",".join(HERE),
       "--scenario-root", str(ROOT), "--cache-dir", str(DATA), "--policies", POLICIES, "--seeds", str(SEEDS),
       "--where", WHERE, "--n-procs", str(N_PROCS)]
proc = subprocess.Popen(cmd, cwd=REPO, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="", flush=True)
print("exit", proc.wait())

In [ ]:
# Against the reference runs, the gate and the zip.
from hvac_controller.d1 import evaluate as ev

results = [json.loads(p.read_text()) for p in sorted(EVAL.glob("*.json"))]
for r in results:
    if "error" in r:
        print("ERROR", r["where"], r["scenario"], r["seed"], r["error"], r["traceback"][-1500:])
ens = ev.ensembles(results + [r for r in REFS if ev.policy_tag(r["policy"]) != TAG])
for where in WHERE.split(","):
    for ref in ("k10", "v1"):
        cmp = ev.compare(ens, TAG, ref, where)
        if not cmp:
            continue
        print(f"\n{TAG} vs {ref} ({where}):")
        for scenario, c in sorted(cmp.items()):
            if scenario != "mean":
                print(f"  {scenario:34s} {c['candidate']:.5f} vs {c['null']:.5f} ({c['rel']:+.1%}, z {c['z']:+.1f})")
        m = cmp["mean"]
        print(f"  mean over {len(cmp) - 1}: {m['candidate']:.5f} vs {m['null']:.5f} ({m['candidate'] / m['null'] - 1:+.1%})")
want = len(HERE) * len(WHERE.split(",")) * SEEDS
print(f"\nGATE: {len(results)} runs (want {want}), no errors:",
      len(results) == want and not any("error" in r for r in results))
name = Path(f"/kaggle/working/score_{TAG}_{RUN_ID}.zip")
with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(EVAL.glob("*.json")):
        z.write(p, p.name)
print("wrote", name)